# 机构调研视角行业配置策略 - 数据获取与预处理

本notebook实现研报《行业配置策略：机构调研视角》的数据获取与预处理模块

**研报核心观点**:
- 机构调研信息反映机构投资者关注方向
- 机构调研次数与股票收益率正相关
- 可构建事件驱动、定期选股、行业轮动三类策略

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from source.data_loader import DataLoader, SurveyDataAggregator
from source.data_preprocessor import DataPreprocessor
from source.analysis import Analysis

## 1. 数据获取

In [ ]:
# 初始化数据加载器
dl = DataLoader()
print("数据加载器初始化成功")

In [ ]:
# 获取中证全指数据作为基准
index_data = dl.get_index_data(index_code='000985.SH', start_date='20150101', end_date='20210228')
print(f"基准指数数据: {len(index_data)} 条记录")
index_data.head()

In [ ]:
# 获取股票基本信息
stock_basic = dl.get_stock_basic_data()
print(f"股票基本信息: {len(stock_basic)} 只股票")
stock_basic.head()

In [ ]:
# 尝试获取机构调研数据
# 注意: 机构调研数据为商业数据,部分接口可能需要付费或数据不可用
print("正在尝试获取机构调研数据...")

survey_df = dl.get_all_survey_data(start_date='20120101', end_date='20210312')
if survey_df is not None:
    print(f"成功获取机构调研数据: {len(survey_df)} 条记录")
else:
    print("警告: 无法获取机构调研数据")
    print("建议: 机构调研数据需要通过Wind、同花顺等商业终端获取")

In [ ]:
# 尝试使用akshare获取替代数据
try:
    import akshare as ak
    survey_df_ak = ak.stock_survey_summary_em()
    print(f"akshare可用, 获取到 {len(survey_df_ak)} 条数据")
except Exception as e:
    print(f"akshare获取失败: {e}")

## 2. 数据预处理

In [ ]:
if survey_df is not None and len(survey_df) > 0:
    preprocessor = DataPreprocessor()
    
    # 清洗数据
    survey_df_clean = preprocessor.clean_survey_data(survey_df)
    print(f"清洗后数据: {len(survey_df_clean)} 条记录")
    
    # 过滤有效股票
    survey_df_valid = preprocessor.filter_valid_stocks(survey_df_clean, stock_basic)
    print(f"有效股票数据: {len(survey_df_valid)} 条记录")
    
    # 对齐交易日
    trading_days = pd.to_datetime(index_data.index)
    survey_df_aligned = preprocessor.align_survey_with_trading_days(survey_df_valid, trading_days)
    print(f"对齐后数据: {len(survey_df_aligned)} 条记录")
else:
    print("无可用数据, 跳过预处理步骤")

## 3. 数据分析

In [ ]:
if survey_df_aligned is not None and len(survey_df_aligned) > 0:
    analysis = Analysis()
    
    # 分析调研分布
    dist_results = analysis.analyze_survey_distribution(survey_df_aligned)
    if dist_results is not None:
        print("调研数据分析结果:")
        print(dist_results.get('daily_stats', pd.DataFrame()).describe())
else:
    print("无可用数据进行统计分析")

## 重要提示

**机构调研数据说明**:

本研报使用的机构调研数据(ASHAREINSTITUTIONALACTIVITY, ASHARINSTITUTIONALPARTICIPANT)为Wind商业数据库专属数据,目前免费数据接口无法获取完整历史数据。

**数据替代方案**:
1. 通过Wind、同花顺、iFind等商业终端获取
2. 使用akshare的实时调研数据作为演示
3. 如有数据来源,请补充到 `source/data_loader.py` 中的 `get_all_survey_data()` 方法